# Qwen Image 2.1 — Setup

Notebook pensato per **Google Colab con A100**.

**Nuova sessione Colab:**
1. seleziona una GPU A100 (e RAM elevata se disponibile);
2. esegui la cella **Installazione**;
3. fai **Runtime → Riavvia sessione**;
4. riparti dalla cella **Carica Qwen Image 2.1**.

Non aggiornare manualmente Torch/CUDA.

In [ ]:
# Controllo GPU (opzionale)
!nvidia-smi

In [ ]:
# Installazione librerie
!pip install -q -U "transformers>=5.17,<5.18" accelerate pillow
!pip install -q -U git+https://github.com/huggingface/diffusers.git
!pip uninstall -y torchao

print("✅ Installazione completata.")
print("Ora: Runtime → Riavvia sessione, poi riparti dalla cella successiva.")

## Carica Qwen Image 2.1

Esegui questa cella **dopo il riavvio della sessione**.

In [ ]:
import torch
from diffusers import QwenImage21Pipeline

pipe = QwenImage21Pipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16
).to("cuda")

print("✅ Pipeline caricata su GPU")
print("GPU:", torch.cuda.get_device_name(0))

# Multi-Reference / Mix di immagini

Usa **da 2 a 10 immagini di riferimento** insieme.

Metti le immagini in:

`MyDrive/QwenMulti/input`

Vengono lette in **ordine alfabetico**. Il prompt può quindi riferirsi a:
“prima immagine”, “seconda immagine”, ecc.

**Importante:** Qwen usa l'ultima immagine come riferimento per il rapporto d'aspetto automatico
dell'output. Se vuoi mettere un vestito su una persona, è comodo usare:

- `01_vestito.png`
- `02_persona.png`

e scrivere: “Put the dress from the first image on the person in the second image...”

In [ ]:
# Collega Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cartelle
import os

BASE_DIR = "/content/drive/MyDrive/QwenMulti"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📥 Metti 2-10 riferimenti qui:", INPUT_DIR)
print("📤 Output:", OUTPUT_DIR)

## Prompt e impostazioni

Esempio vestito + persona:

`Put the dress from the first image on the person in the second image. Preserve the person's identity, face, body pose and background. Preserve the dress design, fabric, colors and details. Make the result photorealistic.`

In [ ]:
PROMPT = '''
Put the dress from the first image on the person in the second image.
Preserve the person's identity, face, body pose and background.
Preserve the dress design, fabric, colors and details.
Make the result photorealistic and naturally fitted.
'''

STEPS = 30
SEED = 42
OUTPUT_RESOLUTION = 1024   # 2048 per output finale
VARIANTS = 1               # aumenta per provare più seed

print("✅ Impostazioni pronte")

In [ ]:
# Genera usando tutti i riferimenti presenti nella cartella input
import os
import json
import torch
from PIL import Image
from datetime import datetime

if "pipe" not in globals():
    raise RuntimeError("❌ Devi prima caricare Qwen Image 2.1.")

extensions = (".png", ".jpg", ".jpeg", ".webp")
ref_files = sorted(
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(extensions)
)

if len(ref_files) < 2:
    raise RuntimeError("❌ Inserisci almeno 2 immagini nella cartella input.")
if len(ref_files) > 10:
    raise RuntimeError("❌ Qwen Image 2.1 supporta fino a 10 immagini di riferimento.")

references = [Image.open(os.path.join(INPUT_DIR, f)).convert("RGB") for f in ref_files]

run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, run_id)
os.makedirs(RUN_OUTPUT_DIR, exist_ok=True)

with open(os.path.join(RUN_OUTPUT_DIR, "prompt.txt"), "w", encoding="utf-8") as f:
    f.write(PROMPT.strip())

with open(os.path.join(RUN_OUTPUT_DIR, "settings.json"), "w", encoding="utf-8") as f:
    json.dump({
        "references": ref_files,
        "steps": STEPS,
        "seed": SEED,
        "output_resolution": OUTPUT_RESOLUTION,
        "variants": VARIANTS,
    }, f, indent=2)

print("🖼️ Riferimenti, nell'ordine:")
for i, name in enumerate(ref_files, 1):
    print(f"  {i}. {name}")
print()

for i in range(VARIANTS):
    current_seed = SEED + i
    print(f"🎨 Variante {i+1}/{VARIANTS} — seed {current_seed}")

    result = pipe(
        prompt=PROMPT,
        image=references,
        num_inference_steps=STEPS,
        output_resolution=OUTPUT_RESOLUTION,
        generator=torch.Generator("cuda").manual_seed(current_seed),
    ).images[0]

    output_path = os.path.join(RUN_OUTPUT_DIR, f"multi_ref_seed_{current_seed}.png")
    result.save(output_path)
    print("✅ Salvata:", output_path)

print("\n🎉 Completato.")